# 4 · Push: data the provider sends as it happens

The negotiation takes two calls here: the `bff-rpc` routes run the intermediate steps on their own. Then the consumer opens a push transfer: the provider subscribes to its backend and the consumer delivers each message to an endpoint of yours, a small HTTP server inside this notebook.

In [ ]:
%run ./02_catalog.ipynb

In [ ]:
steps = []   # every DSP step of this notebook, for the timeline at the end


def rpc(base, path, payload, title=None):
    """POST one DSP step to an agent, show the call and keep it for the timeline."""
    response = requests.post(base + path, headers=json_header, data=json.dumps(payload))
    body = show(response, title)
    steps.append({
        "agent": helpers.AGENTS.get(base, base),
        "step": path.split("/dsp/current/", 1)[-1],
        "state": str(helpers.state_of(body) or "").rsplit(":", 1)[-1],
        "ms": int(response.elapsed.total_seconds() * 1000),
    })
    return body


def timeline(title):
    table(steps, [("Agent", "agent"), ("Step", "step"), ("State", "state"), ("ms", "ms")], title=title)


# The offer goes back exactly as the catalog advertises it.
offer = {
    "@id": policy_id,
    "@type": "Offer",
    "target": target_id,
    "permission": policy.get("permission") or [],
    "obligation": policy.get("obligation") or [],
    "prohibition": policy.get("prohibition") or [],
}

In [ ]:
if "HTTP_PUSH" not in formats(dataset):
    raise RuntimeError(f"{dataset.get('title')} offers no HTTP_PUSH distribution: {formats(dataset)}. Pick another with --dataset.")

payload = {
    "associatedAgentPeer": provider_did,
    "providerAddress": provider_dsp,
    "callbackAddress": consumer_dsp,
    "offer": offer,
}
body = rpc(consumer, "/dsp/current/negotiations/bff-rpc/setup-request-init", payload, "Contract request")
pids = {"consumerPid": body["response"]["consumerPid"], "providerPid": body["response"]["providerPid"]}
body = rpc(provider, "/dsp/current/negotiations/bff-rpc/setup-agreement", pids, "Agreement, and the rest on its own")
agreement_id = body["negotiationAgentModel"]["agreement"]["id"]

## An endpoint for the pushed data

The consumer's agent POSTs every message here. It runs in its container, so it reaches this notebook at `host.docker.internal`, on this session's own port.

In [ ]:
import queue
import threading
import time
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

received_q = queue.Queue()


class Sink(BaseHTTPRequestHandler):
    def do_POST(self):
        raw = self.rfile.read(int(self.headers.get("Content-Length") or 0))
        try:
            received_q.put(json.loads(raw))
        except ValueError:
            received_q.put(raw.decode("utf-8", errors="replace"))
        self.send_response(200)
        self.end_headers()

    def log_message(self, *args):
        pass


sink_port = int(env("EUNOMIA_SINK_PORT"))
sink_url = env("EUNOMIA_SINK_DOCKER_URL")
sink = ThreadingHTTPServer(("0.0.0.0", sink_port), Sink)
threading.Thread(target=sink.serve_forever, daemon=True).start()
print(f"Listening on :{sink_port}; the consumer pushes to {sink_url}")

In [ ]:
payload = {
    "associatedAgentPeer": provider_did,
    "providerAddress": provider_dsp,
    "callbackAddress": consumer_dsp,
    "agreementId": agreement_id,
    "format": "HTTP_PUSH",
    "dataAddress": {"endpointType": "HTTP", "endpoint": sink_url, "endpointProperties": []},
}
body = rpc(consumer, "/dsp/current/transfers/rpc/setup-request", payload, "Push transfer request")
transfer = {"consumerPid": body["response"]["consumerPid"], "providerPid": body["response"]["providerPid"]}
rpc(provider, "/dsp/current/transfers/rpc/setup-start", transfer, "Transfer start: the provider subscribes to its backend");

## The data arrives

The table fills in as messages arrive: five, or 60 seconds.

In [ ]:
received = []
started = time.monotonic()
COLUMNS = [("#", "n"), ("Arrived", "arrived"), ("Message", "message")]
live = table(received, COLUMNS, title="Pushed data, as it arrives")
while len(received) < 5 and time.monotonic() - started < 60:
    try:
        message = received_q.get(timeout=max(0.1, 60 - (time.monotonic() - started)))
    except queue.Empty:
        break
    text = json.dumps(message, ensure_ascii=False) if not isinstance(message, str) else message
    received.append({"n": len(received) + 1, "arrived": f"+{time.monotonic() - started:.1f} s", "message": text, "raw": message})
    table(received, COLUMNS, title=f"Pushed data, as it arrives · {len(received)} so far", handle=live)

if not received:
    raise RuntimeError(f"Nothing arrived. Look at: ./eunomia logs {provider_name}, ./eunomia logs {consumer_name}")
print(json.dumps(received[0]["raw"], indent=2, ensure_ascii=False))

In [ ]:
rpc(provider, "/dsp/current/transfers/rpc/setup-completion", transfer, "Completion: the provider unsubscribes")
sink.shutdown()
sink.server_close()
timeline("Everything this notebook asked of the agents");